In [1]:
import networkx as nx
import numpy as np
#from bngenerator import *
from bngenerator import *
from matplotlib import pyplot as plt

In [120]:
gen = BayesianNetworkGenerator(
        n_nodes=60,
        max_induced_width=12,
        max_degree=20,
        n_iterations=1000
    )

In [121]:
dag = gen.generate().dag

In [123]:
dag.number_of_edges()

202

In [125]:
dag.edges()

OutEdgeView([(1, 2), (1, 23), (1, 4), (2, 3), (2, 4), (2, 53), (3, 53), (3, 4), (3, 45), (3, 52), (4, 5), (4, 44), (4, 13), (4, 18), (4, 24), (4, 16), (4, 20), (4, 6), (4, 12), (5, 6), (8, 14), (8, 18), (8, 12), (8, 59), (8, 5), (9, 10), (9, 11), (9, 6), (10, 6), (11, 12), (11, 15), (11, 13), (11, 14), (11, 26), (11, 17), (11, 16), (12, 13), (12, 43), (12, 15), (12, 26), (13, 14), (13, 16), (13, 26), (14, 15), (14, 59), (14, 5), (14, 10), (15, 16), (15, 5), (16, 17), (17, 18), (17, 54), (17, 59), (17, 5), (19, 20), (19, 54), (19, 53), (19, 3), (20, 57), (20, 40), (20, 56), (20, 46), (20, 41), (20, 54), (21, 22), (21, 42), (22, 23), (22, 16), (22, 33), (22, 5), (22, 11), (22, 35), (22, 32), (22, 7), (22, 26), (22, 17), (22, 31), (23, 24), (23, 29), (23, 7), (23, 37), (23, 31), (23, 12), (23, 30), (24, 25), (24, 16), (24, 17), (24, 28), (24, 11), (24, 26), (24, 14), (24, 8), (25, 15), (27, 28), (27, 8), (27, 14), (27, 13), (28, 29), (28, 0), (28, 17), (28, 15), (28, 6), (28, 16), (28, 13

In [ ]:
INDUCED_WIDTH = [2, 6, 12] #(SPARSE, MEDIUM, DENSE)
CPD_ALPHA = [0.2, 5] # 0.2 = deterministic, 5 = "fuzzy"
NUM_NODES = [10, 20, 30, 40, 50, 60, 80, 100, 150, 200] # H_size = num_nodes * 0.25, following Chen et al.

In [137]:
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD

def generate_controlled_binary_bn(num_nodes=20, induced_width=2, cpd_alpha=5):
    """
    Generates a random binary Bayesian Network with controlled parameters.
    cpd_alpha < 1.0 creates deterministic/rigid networks (closer to 0 or 1).
    cpd_alpha > 1.0 creates fuzzy/uncertain networks (closer to 0.5).
    """

    generator = BayesianNetworkGenerator(n_nodes=num_nodes, 
                                         max_induced_width=induced_width, 
                                         max_degree=20, 
                                         n_iterations=1000)
    
    dag = generator.generate()
    dag = dag.dag
    # Convert to pgmpy format (using string names to avoid int indexing bugs!)
    edges = [(f"V{u}", f"V{v}") for u, v in dag.edges()]
    bn = BayesianNetwork(edges)
    
    # Add isolated nodes if any exist
    for i in range(num_nodes):
        if f"V{i}" not in bn.nodes():
            bn.add_node(f"V{i}")

    # 2. Populate CPDs using the controlled Beta distribution
    for node in bn.nodes():
        parents = list(bn.get_parents(node))
        num_parents = len(parents)
        num_parent_states = 2 ** num_parents
        
        # Draw probabilities from Beta(alpha, alpha)
        # E.g., if alpha=0.1, p_true will mostly be ~0.99 or ~0.01
        p_true = np.random.beta(cpd_alpha, cpd_alpha, size=num_parent_states)
        p_false = 1 - p_true
        
        # TabularCPD expects values as a list of lists: [[P(False)], [P(True)]]
        cpd_values = [p_false.tolist(), p_true.tolist()]
        
        cpd = TabularCPD(
            variable=node,
            variable_card=2,
            values=cpd_values,
            evidence=parents if num_parents > 0 else None,
            evidence_card=[2] * num_parents if num_parents > 0 else None,
            state_names={node: [0, 1], **{p: [0, 1] for p in parents}}
        )
        bn.add_cpds(cpd)
        
    assert bn.check_model()
    return bn

In [138]:
random_bn = generate_controlled_binary_bn()

In [139]:
cpds = random_bn.get_cpds()
for cpd in cpds:
    print(cpd)

+-------+---------------------+---------------------+
| V15   | V15(0)              | V15(1)              |
+-------+---------------------+---------------------+
| V1(0) | 0.65017686109089    | 0.28043107766947695 |
+-------+---------------------+---------------------+
| V1(1) | 0.34982313890911004 | 0.719568922330523   |
+-------+---------------------+---------------------+
+--------+---------------------+-------------------+
| V1     | V1(0)               | V1(1)             |
+--------+---------------------+-------------------+
| V10(0) | 0.7372454249476443  | 0.586034222779784 |
+--------+---------------------+-------------------+
| V10(1) | 0.26275457505235567 | 0.413965777220216 |
+--------+---------------------+-------------------+
+-------+---------------------+-----+--------------------+
| V1    | V1(0)               | ... | V1(1)              |
+-------+---------------------+-----+--------------------+
| V6    | V6(0)               | ... | V6(1)              |
+-------+------